# test_sim — version interactive (sliders)

Copie interactive de `test_sim.ipynb`, réduite aux deux graphes de **rotation vs détuning** et **rotation vs B signé**.
Tous les paramètres se règlent via les sliders / champs numériques ci-dessous ; chaque changement relance le calcul
(deux scans complets ≈ 0,3 s) et redessine :

1. la sphère de Poincaré avec la coupe centrale,
2. la visualisation 3D du champ B,
3. la répartition de population de l'état stationnaire,
4. les deux graphes de rotation (phase et amplitude de frange), en grand.


In [ ]:
import importlib
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from IPython.display import display, clear_output
import module.rb85_bloch as rb85_bloch

importlib.reload(rb85_bloch)


In [ ]:
# ============ outils de fit de frange : I(theta) = c + A*sin(2*theta + phi) ============
theta_values = np.linspace(0, 2 * np.pi, 121)
_design  = np.column_stack([np.ones_like(theta_values),
                            np.sin(2 * theta_values),
                            np.cos(2 * theta_values)])
_design2 = np.column_stack([np.sin(2 * theta_values), np.cos(2 * theta_values)])

def _fit_phase(y):
    """Retourne (phase, amplitude) du fit sin(2*theta+phi) sur y(theta)."""
    c, a, b = np.linalg.lstsq(_design, y, rcond=None)[0]
    A = np.sqrt(a**2 + b**2)
    if A < 1e-30:
        return np.nan, A
    sin_c, cos_c = np.linalg.lstsq(_design2, (y - c) / A, rcond=None)[0]
    return np.arctan2(cos_c, sin_c), A

def _zero_at_zero(x, phase_arr):
    p = np.unwrap(np.where(np.isfinite(phase_arr), phase_arr, np.nan))
    finite = np.isfinite(p)
    if not np.any(finite):
        return p
    return p - np.interp(0, x[finite], p[finite])

def _wrap_pi(phase):
    return (phase + np.pi) % (2 * np.pi) - np.pi

# ============ coupe de Poincaré -> champs d'entrée (Jones) ============
def make_cut(cut_theta, cut_phi):
    """Base (e1, e2) du grand cercle de normale (cut_theta, cut_phi)."""
    n = np.array([np.sin(cut_phi) * np.cos(cut_theta),
                  np.sin(cut_phi) * np.sin(cut_theta),
                  np.cos(cut_phi)])
    n = n / np.linalg.norm(n)
    helper = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(helper, n)) > 0.9:
        helper = np.array([0.0, 1.0, 0.0])
    e1 = np.cross(n, helper)
    e1 = e1 / np.linalg.norm(e1)
    e2 = np.cross(n, e1)
    return e1, e2

def E_in_on_cut(e1, e2):
    """Conversion Poincaré -> Jones le long de la coupe : (n_theta, 2) complexe."""
    E = np.empty((len(theta_values), 2), dtype=complex)
    for i, th in enumerate(theta_values):
        s1, s2, s3 = np.cos(2 * th) * e1 + np.sin(2 * th) * e2
        alpha = np.arccos(np.clip(s1, -1.0, 1.0)) / 2
        delta = np.arctan2(s3, s2)
        E[i] = [np.cos(alpha), np.sin(alpha) * np.exp(1j * delta)]
    return E

component_styles = [
    ('H', '$I_H$', 'tab:blue',   '-'),
    ('V', '$I_V$', 'tab:orange', '-'),
    ('R', '$I_R$', 'tab:green',  '--'),
    ('L', '$I_L$', 'tab:red',    '--'),
    ('D', '$I_D$', 'tab:purple', ':'),
    ('A', '$I_A$', 'tab:brown',  ':'),
]
AMP_THRESHOLD = 1e-2

# ============ calcul complet pour un jeu de paramètres ============
def compute_all(p):
    th, ph = np.radians(p['theta_B_deg']), np.radians(p['phi_B_deg'])
    B_dir = np.array([np.sin(th) * np.cos(ph), np.sin(th) * np.sin(ph), np.cos(th)])
    B_T = p['B_norm_G'] * 1e-4 * B_dir
    pump_det_rad = 2 * np.pi * p['pump_det_MHz'] * 1e6
    gamma_t, n_rho = p['gamma_t'], p['n_rho']
    L_cell = p['L_cell_mm'] * 1e-3

    rho_ss, _ = rb85_bloch.steady_state_rho_closed(B_T, p['pump_s'], pump_det_rad, gamma_t)
    populations = np.real(np.diag(rho_ss))

    e1, e2 = make_cut(np.radians(p['cut_theta_deg']), np.radians(p['cut_phi_deg']))
    E_in = E_in_on_cut(e1, e2)

    # scan en détuning, B fixe
    det_MHz = np.linspace(-p['det_max_MHz'], p['det_max_MHz'], 41)
    J_det = rb85_bloch.jones_matrix_closed(
        2 * np.pi * det_MHz * 1e6, B_T, p['pump_s'], L_cell, n_rho,
        pump_detuning=pump_det_rad, gamma_t=gamma_t,
    )
    I_det = rb85_bloch.polarization_intensities(np.einsum('dij,tj->tdi', J_det, E_in))

    # scan en B signé le long de B_dir, détuning fixe
    B_scan_G = np.linspace(-p['B_max_G'], p['B_max_G'], 25)
    det_fixed = np.array([2 * np.pi * p['det_fixed_MHz'] * 1e6])
    J_B = np.array([
        rb85_bloch.jones_matrix_closed(
            det_fixed, Bn * 1e-4 * B_dir, p['pump_s'], L_cell, n_rho,
            pump_detuning=pump_det_rad, gamma_t=gamma_t,
        )[0]
        for Bn in B_scan_G
    ])
    I_B = rb85_bloch.polarization_intensities(np.einsum('bij,tj->tbi', J_B, E_in))

    def fit_scan(I, x):
        phases, amps = {}, {}
        for key, *_ in component_styles:
            pa = np.array([_fit_phase(I[key][:, j]) for j in range(I[key].shape[1])])
            phases[key] = _zero_at_zero(x, pa[:, 0])
            amps[key] = pa[:, 1]
        return phases, amps

    ph_det, am_det = fit_scan(I_det, det_MHz)
    ph_B,   am_B   = fit_scan(I_B, B_scan_G)

    return dict(B_dir=B_dir, e1=e1, e2=e2, populations=populations,
                det_MHz=det_MHz, ph_det=ph_det, am_det=am_det,
                B_scan_G=B_scan_G, ph_B=ph_B, am_B=am_B)

# ============ tracés ============
def draw_spheres(p, r):
    fig = plt.figure(figsize=(11, 5))

    # sphère de Poincaré + coupe
    ax = fig.add_subplot(121, projection='3d')
    u = np.linspace(0, 2 * np.pi, 40)
    v = np.linspace(0, np.pi, 20)
    ax.plot_surface(np.outer(np.cos(u), np.sin(v)), np.outer(np.sin(u), np.sin(v)),
                    np.outer(np.ones_like(u), np.cos(v)), color='lightgray', alpha=0.12, linewidth=0)
    ax.plot(np.cos(u), np.sin(u), np.zeros_like(u), color='gray', linewidth=1)
    ax.plot(np.cos(u), np.zeros_like(u), np.sin(u), color='gray', linewidth=1)
    ax.plot(np.zeros_like(u), np.cos(u), np.sin(u), color='gray', linewidth=1)
    t = np.linspace(0, 2 * np.pi, 120)
    circle = np.outer(np.cos(t), r['e1']) + np.outer(np.sin(t), r['e2'])
    ax.plot(circle[:, 0], circle[:, 1], circle[:, 2], color='darkred', linewidth=2, label='Coupe centrale')
    for label, (x0, y0, z0) in dict(H=(1, 0, 0), V=(-1, 0, 0), D=(0, 1, 0),
                                    A=(0, -1, 0), R=(0, 0, 1), L=(0, 0, -1)).items():
        ax.scatter([x0], [y0], [z0], color='white', s=120, edgecolors='black', linewidths=0.8)
        ax.text(x0 * 1.15, y0 * 1.15, z0 * 1.15, label, color='black', fontsize=12, ha='center', va='center')
    ax.set_xlim([-1, 1]); ax.set_ylim([-1, 1]); ax.set_zlim([-1, 1])
    ax.set_box_aspect([1, 1, 1])
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title('Sphère de Poincaré')
    ax.legend(loc='upper left', fontsize=8)

    # direction de B et axe de propagation
    ax2 = fig.add_subplot(122, projection='3d')
    ax2.quiver(0, 0, 0, 0, 0, 1, color='purple', label='Propagation (pompe + sonde)')
    Bv = r['B_dir'] * max(p['B_norm_G'], 1e-9)
    ax2.quiver(0, 0, 0, Bv[0], Bv[1], Bv[2], color='blue', label=f"B ({p['B_norm_G']:.2f} G)")
    lim = max(1.0, p['B_norm_G'])
    ax2.set_xlim([-lim, lim]); ax2.set_ylim([-lim, lim]); ax2.set_zlim([-lim, lim])
    ax2.set_box_aspect([1, 1, 1])
    ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
    ax2.set_title('Champ B dans le repère (e1, e2, k)')
    ax2.legend(loc='upper left', fontsize=8)

    fig.tight_layout()
    plt.show()

def draw_populations(p, r):
    ground = r['populations'][:rb85_bloch.N_G]
    excited = r['populations'][rb85_bloch.N_G:]
    fig, (axg, axe) = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    for ax, m_values, pop, title, color in (
        (axg, rb85_bloch.MF3, ground, 'État fondamental (F = 3)', '#2a78d6'),
        (axe, rb85_bloch.MFP4, excited, "État excité (F' = 4)", '#eb6834'),
    ):
        ax.bar(m_values, pop, width=0.6, color=color, zorder=3)
        top = int(np.argmax(pop))
        ax.annotate(f'{pop[top]:.3f}', (m_values[top], pop[top]),
                    textcoords='offset points', xytext=(0, 4), ha='center', fontsize=9)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel(r'$m_F$')
        ax.set_xticks(m_values)
        ax.grid(axis='y', alpha=0.3)
        ax.set_axisbelow(True)
    axg.set_ylabel('Population')
    fig.suptitle(f"Répartition de population (état stationnaire),  B = {p['B_norm_G']:.2f} G,  "
                 f"s = {p['pump_s']:.2f}   —   F=3 : {ground.sum():.3f},  F'=4 : {excited.sum():.3f}",
                 fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    plt.show()

def draw_rotation(p, r):
    fig, axs = plt.subplots(2, 2, figsize=(15, 9), sharex='col',
                            gridspec_kw={'height_ratios': [2, 1]})
    fig.suptitle(f"Rotation (phase de frange) et amplitude  —  s = {p['pump_s']:.2f},  "
                 f"gauche : B = {p['B_norm_G']:.2f} G  |  droite : détuning = {p['det_fixed_MHz']:g} MHz",
                 fontsize=13)

    for col, (x, phases, amps, xlabel) in enumerate([
        (r['det_MHz'], r['ph_det'], r['am_det'], 'Détuning (MHz)'),
        (r['B_scan_G'], r['ph_B'], r['am_B'], 'B signé (G)'),
    ]):
        ax_ph, ax_am = axs[0, col], axs[1, col]
        for key, label, color, ls in component_styles:
            kw = dict(marker='o', markersize=3, linewidth=1.8, color=color, linestyle=ls, label=label)
            if np.nanmean(amps[key]) > AMP_THRESHOLD:
                ax_ph.plot(x, np.degrees(_wrap_pi(phases[key])), **kw)
            ax_am.plot(x, amps[key], **kw)
        for ax in (ax_ph, ax_am):
            ax.axhline(0, color='black', linewidth=0.8, alpha=0.45)
            ax.axvline(0, color='black', linewidth=0.8, alpha=0.45)
            ax.grid(alpha=0.25)
            ax.legend(frameon=False, ncol=3, fontsize=9)
        ax_ph.set_ylabel(r'$\phi - \phi(0)$ mod $2\pi$ (deg)')
        ax_am.set_ylabel('Amplitude $A$')
        ax_am.set_xlabel(xlabel)
        ax_am.set_xlim(x.min(), x.max())

    axs[0, 0].set_title('Rotation vs détuning (B fixe)', fontsize=12)
    axs[0, 1].set_title('Rotation vs B signé (détuning fixe)', fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()


In [ ]:
# ============ sliders / entrées numériques ============
_style = {'description_width': '130px'}
_layout = widgets.Layout(width='340px')

W = dict(
    pump_s        = widgets.FloatLogSlider(value=0.6, base=10, min=-2, max=1.3, step=0.02,
                                           description='s = I/Isat', readout_format='.3f',
                                           continuous_update=False, style=_style, layout=_layout),
    pump_det_MHz  = widgets.FloatSlider(value=0.0, min=-30, max=30, step=0.5,
                                        description='Détuning pompe (MHz)',
                                        continuous_update=False, style=_style, layout=_layout),
    gamma_t       = widgets.FloatLogSlider(value=3.0e4, base=10, min=3, max=6, step=0.05,
                                           description='gamma_t (rad/s)', readout_format='.2e',
                                           continuous_update=False, style=_style, layout=_layout),
    n_rho         = widgets.FloatText(value=5.383e13, description='n (m^-3)',
                                      style=_style, layout=_layout),
    L_cell_mm     = widgets.FloatText(value=75.0, description='L cellule (mm)',
                                      style=_style, layout=_layout),

    B_norm_G      = widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.05,
                                        description='|B| (G)', readout_format='.2f',
                                        continuous_update=False, style=_style, layout=_layout),
    theta_B_deg   = widgets.FloatSlider(value=0.0, min=0.0, max=180.0, step=5.0,
                                        description='B : angle vs k (deg)',
                                        continuous_update=False, style=_style, layout=_layout),
    phi_B_deg     = widgets.FloatSlider(value=0.0, min=0.0, max=360.0, step=5.0,
                                        description='B : azimut (deg)',
                                        continuous_update=False, style=_style, layout=_layout),

    cut_theta_deg = widgets.FloatSlider(value=0.0, min=0.0, max=360.0, step=5.0,
                                        description='Coupe : azimut (deg)',
                                        continuous_update=False, style=_style, layout=_layout),
    cut_phi_deg   = widgets.FloatSlider(value=90.0, min=0.0, max=180.0, step=5.0,
                                        description='Coupe : polaire (deg)',
                                        continuous_update=False, style=_style, layout=_layout),
    det_max_MHz   = widgets.FloatText(value=20.0, description='Scan détuning ± (MHz)',
                                      style=_style, layout=_layout),
    B_max_G       = widgets.FloatText(value=1.0, description='Scan B ± (G)',
                                      style=_style, layout=_layout),
    det_fixed_MHz = widgets.FloatSlider(value=0.0, min=-30, max=30, step=0.5,
                                        description='Détuning du scan B (MHz)',
                                        continuous_update=False, style=_style, layout=_layout),
)

_status = widgets.HTML(value='')
_out = widgets.Output()

def _refresh(change=None):
    p = {k: w.value for k, w in W.items()}
    _status.value = '<i>calcul en cours…</i>'
    with _out:
        clear_output(wait=True)
        import time
        t0 = time.time()
        r = compute_all(p)
        draw_spheres(p, r)
        draw_populations(p, r)
        draw_rotation(p, r)
        _status.value = f'à jour  ({time.time() - t0:.2f} s)'

for w in W.values():
    w.observe(_refresh, names='value')

controls = widgets.HBox([
    widgets.VBox([widgets.HTML('<b>Pompe & cellule</b>'),
                  W['pump_s'], W['pump_det_MHz'], W['gamma_t'], W['n_rho'], W['L_cell_mm']]),
    widgets.VBox([widgets.HTML('<b>Champ B</b>'),
                  W['B_norm_G'], W['theta_B_deg'], W['phi_B_deg']]),
    widgets.VBox([widgets.HTML('<b>Coupe & scans</b>'),
                  W['cut_theta_deg'], W['cut_phi_deg'], W['det_max_MHz'],
                  W['B_max_G'], W['det_fixed_MHz']]),
])

display(widgets.VBox([controls, _status, _out]))
_refresh()
